# Task 2 — Vehicle Complaint EDA and Text Mining

This notebook profiles the supplied complaint dataset, preserves source records, mines narrative entities, clusters failure descriptions, and writes a stakeholder report. Run from the project root after installing `requirements.txt`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from src.text_mining import data_quality, enrich_text, parse_list_cell
from src.pipeline import build_summary, make_figures, write_stakeholder_report

INPUT = Path('data/raw/Dataset.csv')
OUTPUT = Path('outputs')
OUTPUT.mkdir(exist_ok=True)
raw = pd.read_csv(INPUT, low_memory=False)
raw.shape

## 1. Schema, missing data, duplicates, and statistics
Inspect types and null counts first. Blank structured tags are retained and explicitly assessed; they are not fabricated. Dates are parsed with coercion.

In [ ]:
quality = data_quality(raw)
display(pd.DataFrame({'dtype': quality['dtypes'], 'missing': quality['missing_by_column']}))
print('Exact duplicate rows:', quality['exact_duplicate_rows'])
print('Duplicate event IDs:', quality['duplicate_event_ids'])
numeric = raw.select_dtypes(include='number')
display(numeric.describe().T[['count', 'mean', 'std', '50%']])
for col in ['Opened date', 'BUILD_DATE', 'IN_USE_DATE']:
    if col in raw:
        raw[col + '_parsed'] = pd.to_datetime(raw[col], errors='coerce', utc=True)

## 2. Narrative entity tagging and issue categories
Customer, causal, and correction verbatims form the analysis text. Rule-based component, symptom, and repair terms provide interpretable tags; issue categories are coarse screening labels.

In [ ]:
enriched, cluster_terms = enrich_text(raw, cluster_count=6)
display(enriched[['Event id', 'mined_components', 'mined_symptoms', 'mined_repairs', 'issue_category']].head())
display(enriched['issue_category'].value_counts())

## 3. Pattern analysis and clustering
TF-IDF and K-Means group similar complaint narratives. Top terms are descriptive only and do not establish engineering root cause. Compare clusters with technician-reviewed examples before taking action.

In [ ]:
summary = build_summary(enriched)
print('Top components:', summary.get('top_failure_components', {}))
print('Top failure conditions:', summary.get('top_failure_conditions', {}))
print('Cluster terms:', cluster_terms)
make_figures(enriched, OUTPUT / 'figures')
enriched.to_csv(OUTPUT / 'enriched_dataset.csv', index=False)
(OUTPUT / 'data_quality.json').write_text(json.dumps(quality, indent=2), encoding='utf-8')
(OUTPUT / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
(OUTPUT / 'cluster_terms.json').write_text(json.dumps(cluster_terms, indent=2), encoding='utf-8')
write_stakeholder_report(enriched, summary, quality, cluster_terms, OUTPUT / 'stakeholder_report.md')